In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.resnet import ResNet50, preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)
np.random.seed(42)

tf.config.threading.set_intra_op_parallelism_threads(
    tf.config.threading.get_physical_parallelism()
)
tf.config.threading.set_inter_op_parallelism_threads(
    tf.config.threading.get_physical_parallelism()
)



2026-05-05 06:54:15.332775: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777964055.346391      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777964055.350700      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-05 06:54:15.367454: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: module 'tensorflow._api.v2.config.threading' has no attribute 'get_physical_parallelism'

In [2]:
BASE_PATH = "/kaggle/input/cassava-leaf-disease-classification"
TRAIN_IMG_DIR = os.path.join(BASE_PATH, "train_images")
TEST_IMG_DIR = os.path.join(BASE_PATH, "test_images")
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
SAMPLE_SUBMIT = os.path.join(BASE_PATH, "sample_submission.csv")

train_df = pd.read_csv(TRAIN_CSV)
train_df["label"] = train_df["label"].astype(str)  # Keras expects string class names

# Increased batch size to reduce steps per epoch
BATCH_SIZE = 256
IMG_SIZE = (224, 224)
AUTOTUNE = tf.data.AUTOTUNE

train_df["file_path"] = train_df["image_id"].apply(
    lambda x: os.path.join(TRAIN_IMG_DIR, x)
)
label_to_int = {str(i): i for i in range(5)}
train_df["label_int"] = train_df["label"].map(label_to_int)

shuffled = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(0.9 * len(shuffled))
train_split = shuffled.iloc[:split_idx]
valid_split = shuffled.iloc[split_idx:]


def decode_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = preprocess_input(img)
    return img, label


train_ds = tf.data.Dataset.from_tensor_slices(
    (train_split["file_path"].values, train_split["label_int"].values)
)
train_ds = train_ds.map(decode_preprocess, num_parallel_calls=AUTOTUNE)
# Cache decoded images before shuffling to avoid re‑decoding each epoch
train_ds = train_ds.cache()
train_ds = (
    train_ds.shuffle(buffer_size=1024, seed=42).batch(BATCH_SIZE).prefetch(AUTOTUNE)
)

valid_ds = tf.data.Dataset.from_tensor_slices(
    (valid_split["file_path"].values, valid_split["label_int"].values)
)
valid_ds = valid_ds.map(decode_preprocess, num_parallel_calls=AUTOTUNE)
valid_ds = valid_ds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)



2026-05-05 06:54:22.369928: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777964062.370372      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:06:00.0, compute capability: 8.6


In [3]:
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
preds = Dense(5, activation="softmax")(x)  # 5 disease categories (0‑4)
model = Model(inputs=base_model.input, outputs=preds)

for layer in base_model.layers:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

early_stop = EarlyStopping(
    monitor="val_accuracy", patience=7, restore_best_weights=True, mode="max"
)

# Suppressed verbose output to save time
model.fit(
    train_ds,
    epochs=40,
    validation_data=valid_ds,
    callbacks=[early_stop],
    verbose=0,
)

for layer in base_model.layers[-50:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(
    train_ds,
    epochs=30,
    validation_data=valid_ds,
    callbacks=[early_stop],
    verbose=0,
)



94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


I0000 00:00:1777964070.849375     105 service.cc:148] XLA service 0x7f723c0049b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777964070.849407     105 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-05 06:54:31.000119: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777964071.796283     105 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1777964077.923054     105 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-05-05 06:54:59.441028: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5440', 108 bytes spill stores, 108 bytes spill loads

2026-05-05 06:55:07.028435: I external/local_xl

In [4]:
test_files = sorted(
    [f for f in os.listdir(TEST_IMG_DIR) if f.lower().endswith((".jpg", ".png"))]
)
test_df = pd.DataFrame({"image_id": test_files})
test_df["file_path"] = test_df["image_id"].apply(
    lambda x: os.path.join(TEST_IMG_DIR, x)
)

test_ds = tf.data.Dataset.from_tensor_slices(test_df["file_path"].values)
test_ds = test_ds.map(
    lambda p: decode_preprocess(p, 0)[0], num_parallel_calls=AUTOTUNE
)  # ignore label
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

pred_probs = model.predict(test_ds, verbose=0)
pred_labels = np.argmax(pred_probs, axis=1).astype(int)

submission = pd.DataFrame({"image_id": test_df["image_id"], "label": pred_labels})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

2026-05-05 07:01:22.517490: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1677', 3608 bytes spill stores, 3128 bytes spill loads

2026-05-05 07:01:22.914385: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1677', 580 bytes spill stores, 412 bytes spill loads



Submission saved to submission.csv
